In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [10]:
import json
import torch 
from src.config import CHECKPOINT_DIR
from scripts.common.get_device import get_available_device
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT, DATASET_ROOT
from src.Skeleton_model.stgcn import STGCN
from src.XAI.joint_person_occlusion import JointOcclusion
from src.XAI.plotting_functions import plot_joint_occlusion, save_joint_occlusion_video


In [11]:
experiment_root = CHECKPOINT_DIR / "STGCN_V1" / "grid_search_1" / "lr_2e4"
device = get_available_device()

with open(experiment_root / "config.json", "r") as f:
    hyperparameters = json.load(f)

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train", max_people=hyperparameters["max_people"])
val_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="val", max_people=hyperparameters["max_people"])
rgb_dataset = RWF2000Dataset(DATASET_ROOT, split="val", return_rgb_frames=True, num_frames=150)
radii = compute_joint_distance_to_center_of_gravity(train_dataset)
skeleton_graph = SkeletonGraph(radii, normalisation=hyperparameters["adjacency_normalisation_mode"])
model = STGCN(skeleton_graph.A, temporal_kernel_size=hyperparameters["temporal_kernel_size"], dropout=hyperparameters["dropout"], edge_importance_weighting=hyperparameters["edge_importance_weighting"]).to(device)

checkpoint = torch.load(experiment_root / "best_model.pt", map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

Using cuda:3 with 23.35 GB free


/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


STGCN(
  (data_batch_norm): BatchNorm1d(51, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (stgcn_blocks): ModuleList(
    (0): STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(3, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (temporal_conv): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(9, 1), stride=(1, 1), padding=(4, 0), bias=False)
        (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): Dropout(p=0, inplace=False)
      )
      (relu): ReLU(inplace=True)
    )
    (1-3): 3 x STGCNBlock(
      (spatial_graph_conv): SpatialGraphConv(
        (channel_transform): Conv2d(64, 192, kernel_size=(1, 1), stride=(1, 1))
      )
      (temporal_conv): Sequential(
        (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running

In [ ]:
import gc
import torch

output_dir = CHECKPOINT_DIR / "STGCN_V1" / "joint_occlusion_videos"
output_dir.mkdir(parents=True, exist_ok=True)

explainer = JointOcclusion(
    model,
    temporal_window_size=hyperparameters["temporal_kernel_size"],
)

for video_num in range(50):
    print(f"Processing video {video_num + 1}/50")

    input_tensor, label = val_dataset[video_num]
    input_tensor = input_tensor.unsqueeze(0).to(device)

    _, _, rgb_frames = rgb_dataset[video_num]

    results = explainer.explain(input_tensor)

    joint_importance = results["joint_importance"]
    predicted_class = results["predicted_class"]

    save_joint_occlusion_video(
        rgb_frames=rgb_frames,
        pose_tensor=input_tensor,
        joint_importance=joint_importance,
        output_path=output_dir / f"video_{video_num}_pred_{predicted_class}.mp4",
        fps=30,
    )

    del input_tensor, rgb_frames, results, joint_importance
    gc.collect()
    torch.cuda.empty_cache()

Processing video 1/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_0_pred_0.mp4
Processing video 2/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_1_pred_0.mp4
Processing video 3/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_2_pred_1.mp4
Processing video 4/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_3_pred_1.mp4
Processing video 5/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_4_pred_0.mp4
Processing video 6/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusion_videos/video_5_pred_0.mp4
Processing video 7/50
Saved video to: /homes/mp2940/violence-detection-dissertation/checkpoints/STGCN_V1/joint_occlusi